# Microsoft Purview REST API - Fabric Table Metadata Management

This notebook demonstrates how to:
1. Connect to Microsoft Purview using REST APIs
2. Discover a Fabric table
3. Query metadata about the table
4. Update metadata properties

## Prerequisites
- Microsoft Purview account
- Azure credentials with appropriate permissions
- Fabric workspace and table

## 🚀 Quick Start

**Before running this notebook:**

1. **Create `.env` file**: Copy `.env.example` to `.env` and fill in your values
2. **Install packages**: Run Step 1 to install required Python packages
3. **Authenticate**: Follow Step 4 to connect to Azure
4. **Run sequentially**: Execute cells in order from Step 1 to Step 11

**What you'll need:**
- Microsoft Purview account name
- Azure credentials (Azure CLI, browser login, or service principal)
- A Fabric table registered in Purview

See [README.md](README.md) for detailed setup instructions.

## Step 1: Install Required Packages

We'll use the Azure Identity library for authentication and requests for API calls.

# Install required packages


In [ ]:
!pip install azure-identity requests python-dotenv

## Step 2: Import Libraries

In [ ]:
import requests
import json
import os
from azure.identity import DefaultAzureCredential, ClientSecretCredential, InteractiveBrowserCredential
from typing import Optional, Dict, Any
from dotenv import load_dotenv

# Load environment variables from .env file
load_dotenv()

print("✓ Libraries imported and environment variables loaded")

## Step 3: Configure Purview Connection

Set up your Purview account details and authentication.

### Important: Environment Variables Setup

This notebook uses a `.env` file to store sensitive configuration. Before running the notebook:

1. **Create a `.env` file** in the same directory as this notebook (if it doesn't exist)
2. **Copy the content from `.env.example`** as a template
3. **Update the values** in `.env` with your actual Purview account details:
   ```
   PURVIEW_ACCOUNT_NAME=your-actual-purview-account-name
   TABLE_NAME=your-table-name
   ```
4. **Never commit `.env`** to source control (it's already in .gitignore)

The `.env` file keeps your credentials and configuration secure and separate from code.

In [ ]:
# Purview account configuration - loaded from .env file
PURVIEW_ACCOUNT_NAME = os.getenv("PURVIEW_ACCOUNT_NAME")
PURVIEW_ENDPOINT = f"https://{PURVIEW_ACCOUNT_NAME}.purview.azure.com"

# API version
API_VERSION = "2022-08-01-preview"

print(f"Purview Endpoint: {PURVIEW_ENDPOINT}")

if PURVIEW_ACCOUNT_NAME == "your-purview-account-name":
    print("\n⚠ WARNING: Using default Purview account name.")
    print("   Please update PURVIEW_ACCOUNT_NAME in your .env file")

## Step 4: Set Up Authentication

Authenticate using Azure credentials. This uses DefaultAzureCredential which will try multiple authentication methods.

In [ ]:
# OPTION 1: Interactive Browser Authentication (Recommended if Azure CLI not available)
# This will open a browser window for you to sign in
credential = InteractiveBrowserCredential()

# OPTION 2: If you have Azure CLI installed, uncomment this and run 'az login' first:
# credential = DefaultAzureCredential()

# OPTION 3: Service Principal (if you have app credentials in .env)
# Add these to your .env file: AZURE_TENANT_ID, AZURE_CLIENT_ID, AZURE_CLIENT_SECRET
# tenant_id = os.getenv("AZURE_TENANT_ID")
# client_id = os.getenv("AZURE_CLIENT_ID")
# client_secret = os.getenv("AZURE_CLIENT_SECRET")
# if tenant_id and client_id and client_secret:
#     credential = ClientSecretCredential(
#         tenant_id=tenant_id,
#         client_id=client_id,
#         client_secret=client_secret
#     )

# Get access token for Purview
SCOPE = "https://purview.azure.net/.default"

try:
    token = credential.get_token(SCOPE)
    access_token = token.token
    print("✓ Authentication successful")
    print(f"Token expires at: {token.expires_on}")
except Exception as e:
    print(f"✗ Authentication failed: {e}")
    print("\nTroubleshooting:")
    print("1. Make sure you have permissions to access Purview")
    print("2. Try running 'az login' in terminal if Azure CLI is installed")
    print("3. Use InteractiveBrowserCredential() for browser-based login")
    print("4. Check that your .env file has the correct PURVIEW_ACCOUNT_NAME")
    access_token = None

## Step 5: Create Helper Functions

Define helper functions for making API calls to Purview.

In [ ]:
def get_headers() -> Dict[str, str]:
    """Get HTTP headers with authentication token."""
    return {
        "Authorization": f"Bearer {access_token}",
        "Content-Type": "application/json"
    }

def purview_api_call(method: str, endpoint: str, data: Optional[Dict] = None) -> Dict[str, Any]:
    """Make an API call to Purview.
    
    Args:
        method: HTTP method (GET, POST, PUT, DELETE)
        endpoint: API endpoint path
        data: Optional request body for POST/PUT
    
    Returns:
        Response JSON as dictionary
    """
    url = f"{PURVIEW_ENDPOINT}{endpoint}"
    headers = get_headers()
    
    try:
        if method.upper() == "GET":
            response = requests.get(url, headers=headers)
        elif method.upper() == "POST":
            response = requests.post(url, headers=headers, json=data)
        elif method.upper() == "PUT":
            response = requests.put(url, headers=headers, json=data)
        elif method.upper() == "DELETE":
            response = requests.delete(url, headers=headers)
        else:
            raise ValueError(f"Unsupported HTTP method: {method}")
        
        response.raise_for_status()
        return response.json() if response.content else {}
    
    except requests.exceptions.HTTPError as e:
        print(f"HTTP Error: {e}")
        print(f"Response: {response.text}")
        raise
    except Exception as e:
        print(f"Error: {e}")
        raise

print("✓ Helper functions defined")

## Step 6: Search for Fabric Table

Use Purview's search API to discover a Fabric table. You can search by table name, workspace, or other properties.

In [ ]:
# Search parameters - loaded from .env file with fallback defaults
TABLE_NAME = os.getenv("TABLE_NAME")
WORKSPACE_NAME = os.getenv("WORKSPACE_NAME", "")
LAKEHOUSE_NAME = os.getenv("LAKEHOUSE_NAME", "")

# Build search query combining filters
search_terms = []
if TABLE_NAME and TABLE_NAME != "*":
    search_terms.append(TABLE_NAME)
if WORKSPACE_NAME:
    search_terms.append(WORKSPACE_NAME)
if LAKEHOUSE_NAME:
    search_terms.append(LAKEHOUSE_NAME)

SEARCH_QUERY = " ".join(search_terms) if search_terms else "*"

# Filter to only search for Fabric Lakehouse tables
FILTER_TYPE = "fabric_lakehouse_table"

# Build search request
search_payload = {
    "keywords": SEARCH_QUERY,
    "filter": {"entityType": FILTER_TYPE},
    "limit": 25  # Increased limit to see more results
}

print(f"Searching for Fabric Lakehouse tables with query: '{SEARCH_QUERY}'")
print(f"Entity type filter: {FILTER_TYPE}")
print("="*60)

# Execute search
search_endpoint = f"/catalog/api/search/query?api-version={API_VERSION}"
search_results = purview_api_call("POST", search_endpoint, search_payload)

# Display results with entity type information
if search_results.get("value"):
    print(f"Found {len(search_results['value'])} result(s):\n")
    
    # Group by entity type to help identify the correct type
    entity_types = {}
    for asset in search_results["value"]:
        entity_type = asset.get('entityType', 'Unknown')
        if entity_type not in entity_types:
            entity_types[entity_type] = []
        entity_types[entity_type].append(asset)
    
    # Display grouped results
    for entity_type, assets in entity_types.items():
        print(f"\n📂 Entity Type: {entity_type} ({len(assets)} result(s))")
        print("-" * 60)
        for idx, asset in enumerate(assets, 1):
            print(f"{idx}. Name: {asset.get('name', 'N/A')}")
            print(f"   ID: {asset.get('id', 'N/A')}")
            print(f"   Qualified Name: {asset.get('qualifiedName', 'N/A')}")
            # Show additional properties that might help identify it
            if 'assetType' in asset:
                print(f"   Asset Type: {asset.get('assetType')}")
            print()
    
    # Store the first result for next steps
    selected_table = search_results["value"][0]
    table_guid = selected_table.get("id")
    print(f"\n✓ Selected first result - GUID: {table_guid}")
    print(f"  Name: {selected_table.get('name')}")
    print(f"  Type: {selected_table.get('entityType')}")
else:
    print("❌ No results found.")
    print("\nTroubleshooting:")
    print("1. Verify the table name is correct (check Purview portal)")
    print("2. Check if the table is registered in Purview")
    print("3. Ensure you have permissions to view the assets")
    print("4. Verify the table is a Fabric Lakehouse table")
    table_guid = None

## Step 7: Query Table Metadata

Retrieve detailed metadata about the selected Fabric table using its GUID.

In [ ]:
# Use table_guid from Step 6 search results
# If you need to manually set a specific table GUID, uncomment and edit the line below:
# table_guid = "your-table-guid-here"

if 'table_guid' in locals() and table_guid:
    print(f"Fetching metadata for table GUID: {table_guid}")
    print("="*60)
    
    # Get entity details
    entity_endpoint = f"/catalog/api/atlas/v2/entity/guid/{table_guid}?api-version={API_VERSION}"
    entity_data = purview_api_call("GET", entity_endpoint)
    
    # Extract and display key metadata
    if entity_data.get("entity"):
        entity = entity_data["entity"]
        attributes = entity.get("attributes", {})
        
        print("\n📊 Table Information:")
        print(f"  Name: {attributes.get('name', 'N/A')}")
        print(f"  Qualified Name: {attributes.get('qualifiedName', 'N/A')}")
        print(f"  Type: {entity.get('typeName', 'N/A')}")
        print(f"  Description: {attributes.get('description', 'N/A')}")
        print(f"  Owner: {attributes.get('owner', 'N/A')}")
        
        print("\n📝 Additional Attributes:")
        for key, value in attributes.items():
            if key not in ['name', 'qualifiedName', 'description', 'owner']:
                print(f"  {key}: {value}")
        
        # Store for next step
        current_entity = entity
        print("\n✓ Metadata retrieved successfully")
else:
    print("⚠ No table GUID available. Please run Step 6 (Search) first to select a table.")
    current_entity = None

## Step 8: List Table Columns and Descriptions

Retrieve all columns (attributes) of the table with their names, types, and descriptions.

In [ ]:
if table_guid:
    print(f"Fetching columns for table GUID: {table_guid}")
    print("="*60)
    
    # Fetch entity with extended info to get column details
    # Using minExtInfo=true to get related entities (columns)
    columns_endpoint = f"/catalog/api/atlas/v2/entity/guid/{table_guid}?minExtInfo=true&api-version={API_VERSION}"
    entity_with_columns = purview_api_call("GET", columns_endpoint)
    
    columns_found = False
    
    # Method 1: Check relationshipAttributes for columns
    if entity_with_columns.get("entity"):
        entity = entity_with_columns["entity"]
        relationship_attrs = entity.get("relationshipAttributes", {})
        
        # Common column relationship names: "columns", "schema", "attributes"
        column_relationships = relationship_attrs.get("columns") or relationship_attrs.get("schema") or relationship_attrs.get("attributes")
        
        if column_relationships:
            columns_found = True
            print(f"\n📋 Table Columns ({len(column_relationships)} columns):\n")
            print(f"{'#':<4} {'Column Name':<30} {'Data Type':<20} {'Description':<50}")
            print("=" * 105)
            
            for idx, col_ref in enumerate(column_relationships, 1):
                # Column might be a reference or full object
                col_guid = col_ref.get("guid") if isinstance(col_ref, dict) else None
                
                # Try to get column details from referredEntities
                col_details = None
                if col_guid and entity_with_columns.get("referredEntities"):
                    col_details = entity_with_columns["referredEntities"].get(col_guid)
                
                if col_details:
                    col_attrs = col_details.get("attributes", {})
                    col_name = col_attrs.get("name", "N/A")
                    col_type = col_attrs.get("dataType") or col_attrs.get("type") or col_attrs.get("data_type") or "N/A"
                    col_desc = col_attrs.get("description") or col_attrs.get("userDescription") or ""
                    
                    # Truncate description for display
                    col_desc_display = (col_desc[:47] + "...") if len(col_desc) > 50 else col_desc
                    
                    print(f"{idx:<4} {col_name:<30} {str(col_type):<20} {col_desc_display:<50}")
                else:
                    # If we only have the reference
                    col_name = col_ref.get("displayText") or col_ref.get("uniqueAttributes", {}).get("qualifiedName", "N/A")
                    print(f"{idx:<4} {col_name:<30} {'Unknown':<20} {'(Column details not loaded)':<50}")
            
            print()
    
    # Method 2: If columns not in relationships, try searching for related columns
    if not columns_found:
        print("\n⚠ Columns not found in relationshipAttributes. Trying alternative method...")
        
        # Check the entity's attributes for embedded column info
        if current_entity and current_entity.get("attributes"):
            attrs = current_entity.get("attributes", {})
            
            # Look for any attributes that might contain column information
            for key in attrs:
                if "column" in key.lower() or "schema" in key.lower() or "field" in key.lower():
                    print(f"\nFound potential column info in attribute '{key}':")
                    print(f"  {attrs[key]}")
                    columns_found = True
        
        if not columns_found:
            print("\n⚠ Could not retrieve column information using standard methods.")
            print("\nPossible reasons:")
            print("1. Columns may not be indexed in Purview yet")
            print("2. The entity type may store columns differently")
            print("3. You may need additional permissions")
            print("\nTip: Check the Purview portal to see if columns are visible there.")
            print("     If they are, we can inspect the raw response to find where they're stored.")
    
    if columns_found:
        print("\n✓ Column information retrieved successfully")
    
else:
    print("⚠ No table GUID available. Please run the search step first.")

## Step 8b: Prepare Column Descriptions for Update

Create a dictionary of column names and their new descriptions. Modify the descriptions below to match your needs.

In [ ]:
# Extract columns from the previous step and prepare for description updates
column_descriptions = {}

if 'entity_with_columns' in locals() and entity_with_columns.get("entity"):
    entity = entity_with_columns["entity"]
    relationship_attrs = entity.get("relationshipAttributes", {})
    
    # Get column relationships
    column_relationships = relationship_attrs.get("columns") or relationship_attrs.get("schema") or relationship_attrs.get("attributes")
    
    if column_relationships:
        print("📋 Column Descriptions Template")
        print("="*80)
        print("\nExtracted columns from the table. Update the descriptions below:\n")
        
        for col_ref in column_relationships:
            col_guid = col_ref.get("guid") if isinstance(col_ref, dict) else None
            
            # Get column details
            col_details = None
            if col_guid and entity_with_columns.get("referredEntities"):
                col_details = entity_with_columns["referredEntities"].get(col_guid)
            
            if col_details:
                col_attrs = col_details.get("attributes", {})
                col_name = col_attrs.get("name", "N/A")
                col_type = col_attrs.get("dataType") or col_attrs.get("type") or col_attrs.get("data_type") or "N/A"
                current_desc = col_attrs.get("description") or col_attrs.get("userDescription") or ""
                
                # Initialize with current description or placeholder
                column_descriptions[col_name] = {
                    "guid": col_guid,
                    "current_description": current_desc,
                    "new_description": current_desc if current_desc else f"Description for {col_name}",  # Update this
                    "data_type": str(col_type)
                }
        
        # Display the template for user to modify
        print(f"Found {len(column_descriptions)} columns\n")
        print("column_descriptions = {")
        for col_name, col_info in column_descriptions.items():
            print(f'    "{col_name}": {{')
            print(f'        "new_description": "{col_info["new_description"]}",  # ← EDIT THIS')
            print(f'        "data_type": "{col_info["data_type"]}",')
            print(f'        "guid": "{col_info["guid"]}"')
            print(f'    }},')
        print("}\n")
        
        print("✓ Column descriptions prepared")
        print("\n💡 Next steps:")
        print("1. Copy the dictionary above")
        print("2. Create a new cell and paste it")
        print("3. Update the 'new_description' values for each column")
        print("4. Run that cell to set the column_descriptions variable")
        print("5. Then proceed to Step 9 to apply the updates")
        
    else:
        print("⚠ No columns found. Make sure Step 8 ran successfully.")
else:
    print("⚠ No column data available. Please run Step 8 first to fetch column information.")

## Step 9: Update Table and Column Metadata

Update table metadata (description, owner) and column descriptions using the Purview REST API.

In [ ]:
if table_guid and current_entity:
    print(f"Updating metadata for table GUID: {table_guid}")
    print("="*60)
    
    # PART 1: Update Table Metadata
    print("\n📊 Part 1: Updating Table Metadata")
    print("-" * 60)
    
    # Prepare table update payload
    update_payload = {
        "entity": {
            "typeName": current_entity.get("typeName"),
            "attributes": {
                "qualifiedName": current_entity["attributes"]["qualifiedName"],
                "name": current_entity["attributes"]["name"],
                # Update these fields as needed:
                "description": "Updated description via Purview REST API",
                "owner": "your-email@example.com",  # Replace with actual owner
            },
            "guid": table_guid
        }
    }
    
    print("\nUpdating the following table attributes:")
    print(f"  Description: {update_payload['entity']['attributes']['description']}")
    print(f"  Owner: {update_payload['entity']['attributes']['owner']}")
    
    # Execute table update
    update_endpoint = f"/catalog/api/atlas/v2/entity?api-version={API_VERSION}"
    
    try:
        update_result = purview_api_call("POST", update_endpoint, update_payload)
        print("\n✓ Table metadata updated successfully")
        
        # Display updated entity
        if update_result.get("mutatedEntities"):
            for entity_type, entities in update_result["mutatedEntities"].items():
                for entity in entities:
                    print(f"  Updated: {entity.get('typeName')}: {entity.get('attributes', {}).get('name')}")
    
    except Exception as e:
        print(f"\n✗ Table update failed: {e}")
    
    # PART 2: Update Column Descriptions
    print("\n\n📝 Part 2: Updating Column Descriptions")
    print("-" * 60)
    
    if 'column_descriptions' in locals() and column_descriptions:
        print(f"\nPreparing to update {len(column_descriptions)} column(s)\n")
        
        updated_count = 0
        skipped_count = 0
        failed_count = 0
        
        for col_name, col_info in column_descriptions.items():
            # Only update if new_description is different from current
            if col_info.get("new_description") and col_info["new_description"] != col_info.get("current_description"):
                col_guid = col_info.get("guid")
                
                if col_guid:
                    try:
                        # Fetch current column entity to get complete structure
                        col_endpoint = f"/catalog/api/atlas/v2/entity/guid/{col_guid}?api-version={API_VERSION}"
                        col_entity_data = purview_api_call("GET", col_endpoint)
                        
                        if col_entity_data.get("entity"):
                            col_entity = col_entity_data["entity"]
                            
                            # Build update payload preserving all original attributes
                            column_update_payload = {
                                "entity": {
                                    "typeName": col_entity.get("typeName"),
                                    "guid": col_guid,
                                    "attributes": col_entity.get("attributes", {}).copy()
                                }
                            }
                            
                            # Update only the description
                            column_update_payload["entity"]["attributes"]["description"] = col_info["new_description"]
                            
                            # Remove any attributes that might be read-only or cause issues
                            attrs_to_remove = ["createTime", "modifiedTime", "createBy", "modifiedBy"]
                            for attr in attrs_to_remove:
                                column_update_payload["entity"]["attributes"].pop(attr, None)
                            
                            # Execute individual column update
                            try:
                                col_update_result = purview_api_call("POST", update_endpoint, column_update_payload)
                                updated_count += 1
                                print(f"  ✓ Updated: {col_name}")
                                print(f"    → \"{col_info['new_description'][:70]}...\"" if len(col_info['new_description']) > 70 else f"    → \"{col_info['new_description']}\"")
                            
                            except requests.exceptions.HTTPError as http_err:
                                failed_count += 1
                                print(f"  ✗ Failed to update {col_name}: {http_err}")
                                # Try to get more details from the response
                                try:
                                    error_detail = http_err.response.json()
                                    print(f"    Error details: {error_detail}")
                                except:
                                    print(f"    Error response: {http_err.response.text[:200]}")
                    
                    except Exception as e:
                        failed_count += 1
                        print(f"  ✗ Error processing {col_name}: {e}")
                else:
                    skipped_count += 1
                    print(f"  ⚠ Skipped {col_name}: No GUID available")
            else:
                skipped_count += 1
        
        # Summary
        print("\n" + "-" * 60)
        print("📊 Update Summary:")
        print(f"  ✓ Successfully updated: {updated_count} column(s)")
        if skipped_count > 0:
            print(f"  ⊘ Skipped (no changes): {skipped_count} column(s)")
        if failed_count > 0:
            print(f"  ✗ Failed: {failed_count} column(s)")
        
        if updated_count > 0:
            print("\n✓ Column descriptions updated successfully")
        elif failed_count > 0:
            print("\n⚠ Some columns failed to update. Check error messages above.")
        else:
            print("\n💡 No column descriptions needed updating (all match current descriptions)")
    
    else:
        print("\n💡 No column_descriptions variable found. Run Step 8b first to prepare column descriptions.")
        print("   Or, the column_descriptions dictionary is empty.")
    
    print("\n" + "="*60)
    print("✓ Update process complete")
    
else:
    print("⚠ No table GUID or entity available. Please run previous steps first.")

## Step 10: Verify the Updates

Query the metadata again to confirm the table and column changes were applied.

In [ ]:
if table_guid:
    print(f"Verifying updated metadata for table GUID: {table_guid}")
    print("="*60)
    
    # Get updated entity details
    entity_endpoint = f"/catalog/api/atlas/v2/entity/guid/{table_guid}?api-version={API_VERSION}"
    updated_entity_data = purview_api_call("GET", entity_endpoint)
    
    if updated_entity_data.get("entity"):
        entity = updated_entity_data["entity"]
        attributes = entity.get("attributes", {})
        
        print("\n📊 Verified Table Information:")
        print(f"  Name: {attributes.get('name', 'N/A')}")
        print(f"  Description: {attributes.get('description', 'N/A')}")
        print(f"  Owner: {attributes.get('owner', 'N/A')}")
        print(f"  Modified Time: {entity.get('updateTime', 'N/A')}")
        
        print("\n✓ Verification complete")
else:
    print("⚠ No table GUID available. Please run the search step first.")

## Step 11: Display Updated Column Descriptions

Fetch and display all columns with their updated descriptions to confirm successful updates.

In [ ]:
if table_guid:
    print(f"Displaying updated column descriptions for table GUID: {table_guid}")
    print("="*80)
    
    # Fetch fresh column data with descriptions
    columns_endpoint = f"/catalog/api/atlas/v2/entity/guid/{table_guid}?minExtInfo=true&api-version={API_VERSION}"
    fresh_entity_data = purview_api_call("GET", columns_endpoint)
    
    if fresh_entity_data.get("entity"):
        entity = fresh_entity_data["entity"]
        relationship_attrs = entity.get("relationshipAttributes", {})
        
        # Get column relationships
        column_relationships = relationship_attrs.get("columns") or relationship_attrs.get("schema") or relationship_attrs.get("attributes")
        
        if column_relationships:
            print(f"\n✅ Successfully Retrieved Column Descriptions")
            print(f"\nTable: {entity.get('attributes', {}).get('name', 'N/A')}")
            print(f"Total Columns: {len(column_relationships)}\n")
            print("="*80)
            print(f"{'#':<4} {'Column Name':<35} {'Description':<40}")
            print("="*80)
            
            columns_with_desc = []
            columns_without_desc = []
            
            for idx, col_ref in enumerate(column_relationships, 1):
                col_guid = col_ref.get("guid") if isinstance(col_ref, dict) else None
                
                # Get column details from referredEntities
                col_details = None
                if col_guid and fresh_entity_data.get("referredEntities"):
                    col_details = fresh_entity_data["referredEntities"].get(col_guid)
                
                if col_details:
                    col_attrs = col_details.get("attributes", {})
                    col_name = col_attrs.get("name", "N/A")
                    col_desc = col_attrs.get("description") or col_attrs.get("userDescription") or ""
                    
                    if col_desc:
                        columns_with_desc.append((idx, col_name, col_desc))
                    else:
                        columns_without_desc.append((idx, col_name))
            
            # Display columns with descriptions
            for idx, col_name, col_desc in columns_with_desc:
                # Wrap long descriptions
                if len(col_desc) > 40:
                    desc_lines = [col_desc[i:i+40] for i in range(0, len(col_desc), 40)]
                    print(f"{idx:<4} {col_name:<35} {desc_lines[0]:<40}")
                    for desc_line in desc_lines[1:]:
                        print(f"{'':4} {'':35} {desc_line:<40}")
                else:
                    print(f"{idx:<4} {col_name:<35} {col_desc:<40}")
            
            # Display columns without descriptions (if any)
            if columns_without_desc:
                print("\n" + "-"*80)
                print("⚠ Columns without descriptions:")
                print("-"*80)
                for idx, col_name in columns_without_desc:
                    print(f"{idx:<4} {col_name:<35} {'(No description)':<40}")
            
            print("\n" + "="*80)
            print(f"📊 Summary:")
            print(f"  ✓ Columns with descriptions: {len(columns_with_desc)}")
            if columns_without_desc:
                print(f"  ⚠ Columns without descriptions: {len(columns_without_desc)}")
            print(f"  📋 Total columns: {len(column_relationships)}")
            
            print("\n✅ Column descriptions displayed successfully!")
            
        else:
            print("⚠ No column relationships found in the entity.")
    else:
        print("⚠ Could not fetch entity data.")
        
else:
    print("⚠ No table GUID available. Please run the search step first.")